In [3]:
# -*- coding: utf-8 -*-
"""Make_Jokin_Python.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RPfy-4KBMbn76aXD7TXd-DSTtjF_lUAL
"""

"""
従業者の勤務の体制及び勤務形態一覧表 作成処理

内部仕様書 1)〜3) に対応する構成:
  1. 入力読み込み           -> RosterReader
  2. 月ごとに1回で構築       -> FteApp + AttendanceSheetBuilder
  3. 保存                  -> WorkScheduleGenerator
"""

import calendar
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path

import openpyxl


# ---------- 用語（[職種]表） ----------

VALID_ROLE_CODES = {"NS", "PT", "OT"}
ROLE_SHORT_LABEL = {"NS": "看", "PT": "理", "OT": "作"}
ROLE_FULL_LABEL = {"NS": "看護職員", "PT": "理学療法士", "OT": "作業療法士"}


# ---------- データ構造 ----------

@dataclass
class Employee:
    """社員リストの1行分（社員1人分）の情報を持つだけの入れ物"""
    name: str             # 全角スペース区切りのフルネーム 例: "福田　仁美"
    role_code: str         # "NS"（看護職員） / "PT"（理学療法士） / "OT"（作業療法士）
    employment_type: str   # "A"（常勤専従） / "B"（常勤兼務） / "C"（非常勤専従） / "D"（非常勤兼務）


@dataclass
class RosterPeriod:
    """1つの職員構成期間（社員リストの1シート分）の情報を持つ入れ物"""
    year: int                     # 対象年 例: 2026
    start_month: int               # 開始月 例: 1
    end_month: int                  # 終了月 例: 8
    sheet_name: str                  # 社員リスト内のシート名 例: "2026.01-08"
    employees: list[Employee]         # このシートに記載されている社員全員

    @property
    def manager(self) -> Employee:
        """職種がNS(看護職員)かつ雇用区分がBの職員を管理者として返す"""
        for employee in self.employees:
            if employee.role_code == "NS" and employee.employment_type == "B":
                return employee
        raise ValueError(
            f"{self.sheet_name}: 職種NS・雇用区分Bの職員が見つかりません（管理者を決定できません）"
        )


# ---------- 1. 入力読み込み ----------

class RosterReader:
    """社員リスト.xlsx を読み込み、RosterPeriod のリストを作る"""

    SHEET_LIST_NAME = "作成シートリスト"

    def __init__(self, path):
        self.wb = openpyxl.load_workbook(path, data_only=False)

    def read_periods(self) -> list[RosterPeriod]:
        """作成シートリストに載っている期間を、上から順に全て読み込む"""
        list_ws = self.wb[self.SHEET_LIST_NAME]
        periods = []
        row = 3  # A3が最初の期間（A2は見出し）
        while list_ws.cell(row=row, column=1).value is not None:
            sheet_name = list_ws.cell(row=row, column=1).value
            periods.append(self._read_period(sheet_name))
            row += 1
        return periods

    def _read_period(self, sheet_name: str) -> RosterPeriod:
        """1つの社員構成シート（例 "2026.01-08"）から RosterPeriod を作る"""
        ws = self.wb[sheet_name]
        return RosterPeriod(
            year=ws["B16"].value,
            start_month=self._parse_month_cell(ws["B17"].value),
            end_month=self._parse_month_cell(ws["C17"].value),
            sheet_name=sheet_name,
            employees=self._read_employees(ws),
        )

    def _read_employees(self, ws) -> list[Employee]:
        """社員表（A4行目以降、A列が空になるまで）を読み込む"""
        employees = []
        row = 4  # A4が最初の社員（A3は見出し）
        while ws.cell(row=row, column=1).value is not None:
            name = ws.cell(row=row, column=1).value
            role_code = ws.cell(row=row, column=2).value
            employment_type = ws.cell(row=row, column=3).value

            if role_code not in VALID_ROLE_CODES:
                raise ValueError(
                    f"{ws.title}: {name} の職種コード {role_code!r} が不正です"
                    f"（NS/PT/OTのいずれかである必要があります）"
                )

            employees.append(Employee(name, role_code, employment_type))
            row += 1
        return employees

    @staticmethod
    def _parse_month_cell(value) -> int:
        """
        対象月セルの値を整数の月番号にする。
        Excel側でセルが日付書式になっていると、openpyxlはdatetimeとして
        値を返してくる（例: 8 → 1900-01-08）ので、その場合は.dayを使う。
        """
        if isinstance(value, datetime):
            return value.day
        return int(value)


def validate_periods_contiguous(periods: list[RosterPeriod]) -> None:
    """期間同士が年をまたがず連続していることを検証する"""
    for prev, curr in zip(periods, periods[1:]):
        if prev.year != curr.year:
            raise ValueError(f"{prev.sheet_name} → {curr.sheet_name}: 年をまたいでいます")
        if prev.end_month + 1 != curr.start_month:
            raise ValueError(
                f"{prev.sheet_name}(終了月{prev.end_month}) と "
                f"{curr.sheet_name}(開始月{curr.start_month}) が連続していません"
            )


# ---------- 常勤換算算出アプリ側 ----------

class FteApp:
    """
    常勤換算算出アプリ.xlsm から月次の勤務実績を取り出す。

    注意: 管理者(職種NS・雇用区分B)だけは、アプリ側のシートに
    まったく同じ内容の行が2行存在する。目的シート側も管理者用に
    2行（11行目=管理者、12行目=看護職員本人）必要なので、行の並び順
    そのままコピーすれば、特別扱いしなくても自然に2行とも埋まる。
    """

    SOURCE_TOP_ROW = 2      # 実績データの先頭行（アプリ側）
    SOURCE_ROW_COUNT = 15   # 2〜16行目の15行分
    SOURCE_LEFT_COL = 2     # B列（1日目）

    def __init__(self, path):
        self.wb = openpyxl.load_workbook(path, data_only=True, keep_vba=True)

    def has_month(self, month: int) -> bool:
        return str(month) in self.wb.sheetnames

    def get_daily_cells(self, year: int, month: int):
        """
        該当月シートの日々実績値ブロック（15行 × その月の日数分の列）を、
        openpyxlのCellオブジェクトの2次元リスト(行ごとのリストのリスト)で返す。
        該当月のシートが無ければ None を返す。
        """
        if not self.has_month(month):
            return None

        ws = self.wb[str(month)]
        days = calendar.monthrange(year, month)[1]

        rows = []
        for r in range(self.SOURCE_ROW_COUNT):
            row_cells = [
                ws.cell(row=self.SOURCE_TOP_ROW + r, column=self.SOURCE_LEFT_COL + c)
                for c in range(days)
            ]
            rows.append(row_cells)
        return rows


# ---------- 出力シート構築 ----------

class AttendanceSheetBuilder:
    """テンプレートを複製した1シート分の書き込みを担当"""

    HEADER_ERA_CELL = "H1"     # 例: "令和8"
    HEADER_MONTH_CELL = "L1"   # 例: 1

    DATE_ROW = 9
    DATE_COL_START = 5   # E列 = 1日目
    MAX_DAYS = 31         # 第5週まで含めた最大列数

    DATA_ROW_START = 11   # 管理者行
    DATA_ROW_END = 25     # 作業療法士の最終行

    # 役割ごとに割り当てる出力シートの行範囲（内部仕様書3)より）
    ROLE_ROW_RANGES = {
        "NS": (12, 21),
        "PT": (22, 22),
        "OT": (23, 25),
    }

    def __init__(self, worksheet):
        self.ws = worksheet

    def set_header(self, year: int, month: int) -> None:
        """1行目の令和年・月を書き込む"""
        reiwa_year = year - 2018
        self.ws[self.HEADER_ERA_CELL] = f"令和{reiwa_year}"
        self.ws[self.HEADER_MONTH_CELL] = month

    def set_date_row(self, year: int, month: int) -> None:
        """
        日付行(9行目)を書き込む。曜日行(10行目)は数式が日付を見て
        自動計算するので触らない。日数が31日に満たない月は、
        余った列（第5週など）を明示的に空欄にする。
        """
        days = calendar.monthrange(year, month)[1]
        for d in range(1, self.MAX_DAYS + 1):
            col = self.DATE_COL_START + d - 1
            cell = self.ws.cell(row=self.DATE_ROW, column=col)
            cell.value = date(year, month, d) if d <= days else None

    def set_roster(self, period: RosterPeriod) -> None:
        """職員構成(A〜D列)を役割ごとの固定行へ書き込む"""
        # 前の月のテンプレート内容が残らないよう、まず全体をクリアする
        for row in range(self.DATA_ROW_START, self.DATA_ROW_END + 1):
            for col in range(1, 5):  # A,B,C,D
                self.ws.cell(row=row, column=col).value = None

        manager = period.manager
        ns_employees = [e for e in period.employees if e.role_code == "NS"]
        pt_employees = [e for e in period.employees if e.role_code == "PT"]
        ot_employees = [e for e in period.employees if e.role_code == "OT"]
        remaining_ns = [e for e in ns_employees if e is not manager]

        # 11行目: 管理者（12行目と同一人物・同一内容）
        self._write_employee_row(self.DATA_ROW_START, "管理者", manager)

        # 12行目以降: 管理者本人(看護職員として) → 残りの看護職員
        self._write_role_block(*self.ROLE_ROW_RANGES["NS"], [manager] + remaining_ns)
        self._write_role_block(*self.ROLE_ROW_RANGES["PT"], pt_employees)
        self._write_role_block(*self.ROLE_ROW_RANGES["OT"], ot_employees)

    def write_actual_hours(self, fte_app: FteApp, year: int, month: int) -> None:
        """
        常勤換算算出アプリの該当月シートから、日々の実績値を書き込む。
        該当月のシートが無ければ空欄のままにする。
        週合計・週平均・常勤換算の数式列(AJ〜AL)は対象外で、上書きしない。

        値だけをコピーし、罫線・数値書式などのセルの書式は変更しない。
        E11:AI25はテンプレート（サンプルシート）を複製した時点で、
        既に罫線・数値書式（負数を赤字にする等）が正しく設定されている。
        常勤換算アプリ側のセルには罫線が無く数値書式も「標準」なので、
        もし書式までコピーしてしまうと、テンプレート側の罫線が消えてしまう。
        """
        days = calendar.monthrange(year, month)[1]

        # 1. まずテンプレートの値(サンプル月のもの)を全部クリアする（書式はそのまま）
        for row in range(self.DATA_ROW_START, self.DATA_ROW_END + 1):
            for col_offset in range(self.MAX_DAYS):
                self.ws.cell(row=row, column=self.DATE_COL_START + col_offset).value = None

        # 2. アプリ側にデータが無ければ、空欄のままここで終わり
        src_rows = fte_app.get_daily_cells(year, month)
        if src_rows is None:
            return

        # 3. その月の日数分だけ、値だけをコピーする
        for row_offset, src_row_cells in enumerate(src_rows):
            dst_row = self.DATA_ROW_START + row_offset
            for col_offset in range(days):
                src_cell = src_row_cells[col_offset]
                dst_cell = self.ws.cell(row=dst_row, column=self.DATE_COL_START + col_offset)
                dst_cell.value = src_cell.value

    def _write_role_block(self, start_row: int, end_row: int, employees: list[Employee]) -> None:
        capacity = end_row - start_row + 1
        if len(employees) > capacity:
            raise ValueError(
                f"{ROLE_FULL_LABEL.get(employees[0].role_code, '職員')}が{len(employees)}名おり、"
                f"テンプレートの枠({capacity}行)を超えています"
            )
        for i, employee in enumerate(employees):
            role_label = ROLE_FULL_LABEL[employee.role_code]
            self._write_employee_row(start_row + i, role_label, employee)

    def _write_employee_row(self, row: int, role_label: str, employee: Employee) -> None:
        self.ws.cell(row=row, column=1).value = role_label                         # A: 職種
        self.ws.cell(row=row, column=2).value = ROLE_SHORT_LABEL[employee.role_code]  # B: 資格(略号)
        self.ws.cell(row=row, column=3).value = employee.employment_type            # C: 勤務形態
        self.ws.cell(row=row, column=4).value = employee.name                       # D: 氏名


# ---------- 司令塔 ----------

class WorkScheduleGenerator:
    """全期間・全月を処理して、出力ファイルを作る"""

    SAMPLE_SHEET_NAME = "勤務形態一覧表（訪問看護） 2026.01実績(サンプル)"

    def __init__(self, roster_path, template_path, app_path, out_dir="."):
        self.roster_path = Path(roster_path)
        self.template_path = Path(template_path)
        self.app_path = Path(app_path)
        self.out_dir = Path(out_dir)

    def run(self) -> None:
        """
        作成シートリストの全期間分をひとまとめにして、
        常に1つのファイルとして出力する。
        """
        periods = RosterReader(self.roster_path).read_periods()
        validate_periods_contiguous(periods)

        fte_app = FteApp(self.app_path)

        out_wb = openpyxl.load_workbook(self.template_path)
        sample_ws = out_wb[self.SAMPLE_SHEET_NAME]

        for period in periods:
            self._add_period_sheets(out_wb, sample_ws, period, fte_app)

        out_wb.remove(sample_ws)  # テンプレートのサンプルシートは出力に含めない

        out_path = self.out_dir / self._build_filename(periods)
        out_wb.save(out_path)
        print(f"作成しました: {out_path}")

    def _add_period_sheets(self, out_wb, sample_ws, period: RosterPeriod, fte_app: FteApp) -> None:
        """1つの期間分の月シートを、既存の出力ブックに追加していく"""
        for month in range(period.start_month, period.end_month + 1):
            new_ws = out_wb.copy_worksheet(sample_ws)
            new_ws.title = f"勤務形態一覧表（訪問看護） {period.year}.{month:02d}実績"

            builder = AttendanceSheetBuilder(new_ws)
            builder.set_header(period.year, month)
            builder.set_date_row(period.year, month)
            builder.set_roster(period)
            builder.write_actual_hours(fte_app, period.year, month)

            if not fte_app.has_month(month):
                print(f"  警告: {period.year}年{month}月は常勤換算アプリにデータが無いため、勤務実績は空欄です")

    @staticmethod
    def _build_filename(periods: list[RosterPeriod]) -> str:
        """
        全期間をまとめた1ファイルのファイル名を作る。
        YY = 最初の期間の年、XX = 最初の期間の開始月、ZZ = 最後の期間の終了月
        （period同士は年をまたがず連続している前提: validate_periods_contiguousで検証済み）
        """
        year = periods[0].year
        start_month = periods[0].start_month
        end_month = periods[-1].end_month
        return (
            f"従業者の勤務の体制及び勤務形態一覧表_"
            f"{year}年{start_month:02d}月-{end_month:02d}月実績.xlsx"
        )


# ---------- 実行 ----------

if __name__ == "__main__":
    generator = WorkScheduleGenerator(
        roster_path="社員リスト.xlsx",
        template_path="従業者の勤務の体制及び勤務形態一覧表_20YY年XX月-ZZ月実績.xlsx",
        app_path="常勤換算算出アプリ_汎用型_260523.xlsm",
        out_dir=".",
    )
    generator.run()

  警告: 2026年8月は常勤換算アプリにデータが無いため、勤務実績は空欄です
  警告: 2026年9月は常勤換算アプリにデータが無いため、勤務実績は空欄です
  警告: 2026年10月は常勤換算アプリにデータが無いため、勤務実績は空欄です
  警告: 2026年11月は常勤換算アプリにデータが無いため、勤務実績は空欄です
  警告: 2026年12月は常勤換算アプリにデータが無いため、勤務実績は空欄です
作成しました: 従業者の勤務の体制及び勤務形態一覧表_2026年01月-12月実績.xlsx
